### Welcome

This is the first section of the introductory GeoIPS tutorial, which includes running GeoIPS 
using the CLI, and creating your first plugin! 




### Tutorial Scope

This tutorial does not address running GeoIPS in near real-time. GeoIPS plugins 
are intended to be developed and tested on a specific dataset, and setting up the 
real-time processing infrastructure is a separate conversation. 

This tutorial focuses on: 

  - Workflow / Algorithm development and testing 
  - Running GeoIPS 


### Installing Appropriate Test Datasets

For this tutorial, you'll need GeoIPS test datasets that can be used to produce imagery 
or scientific datasets. 

If just attending the beginner tutorial, you'll need:

  - `test_data_abi`

These datasets might take a while to download so go ahead and download the appropriate 
datasets based on the following commands.

**NOTE:** these datasets may already be installed on the JupyterHub instance.

In [ ]:
%%bash

# NOTE: UNCOMMENT THE LINE BELOW IF THAT DATASET HAS NOT YET BEEN INSTALLED
# geoips config install test_data_abi
pluginify config set rebuild-registries True

## Restart Your JupyterLab Kernel

To ensure JupyterLab recognizes the new configuration variable you just set, restart
your kernel.

### Introduction to GeoIPS

GeoIPS is a plugin-based system for processing geolocated data: 
  - produce imagery in several formats (most often PNG). 
  - produce output data products in NetCDF4 format. 
  - extended to add other output formats via plugins. 

![1111𝝁 infrared imagery](./images/conus_infrared.png)

![Himawari-9 CLAVR-x Cloud-Top-Height](./images/ahi_cloud_top_height.png)

![ABI CLAVR-x Cloud Top Height](./images/abi_cloud_top_height.png)

GeoIPS is almost entirely composed of plugins:

  - GeoIPS can be extended by developing new plugins in external python packages. 
  - No need to edit the main GeoIPS code to add new functionality. 
  - Most types of functionality in GeoIPS can be extended (and if something can’t be 
  extended, and you think it should, let us know!). 


### Vocabulary

**YAML**
  - "Yet Another Markdown Language" 
  - A human-readable data serialization language that is often used for writing configuration files 

**Plugin**
  - A Python module storing a plugin object or YAML file that defines GeoIPS functionality 
  - Stored in an installable Python package that registers its plugin payload with GeoIPS 

**Interface**
  - A class of Python plugins that modify the same type of functionality within GeoIPS 
  (e.g., “the algorithms interface” or “the colormappers interface”) 


### Plugin Types Highlighted Today

**algorithm**
  - Implements a function that modifies data and outputs new data

**procflow**
  - Executes a series of steps (plugins) in the order specified in a workflow plugin.
  - Operates on a xarray.DataTree object to store the data and metadata from each step
    in a tree-like fashion
  - Transforms data to expected type for a given plugin and back to a DataTree after 
    that plugin has been executed

**workflow**
  - Defines the order of operations in which a procflow will execute

For a listing of all plugin types and information about each, please visit
[GeoIPS Interfaces Documentation](https://nrlmmd-geoips.github.io/geoips/concepts/functionality/interfaces/index.html).

## Hands On: Modify a plugin template to create your own installable plugin package  


### Get the Template Repository

To add new plugins to GeoIPS, we use Plugin Packages. You will use a template repository to build your own, installable plugin package containing custom plugins.

The following commands will clone the template plugin repository. Typically, when staring a new plugin package, you would choose the name of your package. We've taken that joy from you and have named your new package "cool_plugins". This name is stored in `$MY_PKG_NAME` for later use. 

`$MY_PKG_DIR` is also provided for convenience and contains the full path to your new package.



In [ ]:
%%bash

cd $GEOIPS_PACKAGES_DIR

if [[ ! -d "template_basic_plugin" && ! -d "$MY_PKG_NAME" ]]; then
    echo "Cloning template_basic_plugin package"
    git clone --no-tags --single-branch $GEOIPS_REPO_URL/template_basic_plugin.git
else
    echo "Package already exists"
fi

# Rename your package
if [[ ! -d "$MY_PKG_NAME" ]]; then
    echo "Renaming template_basic_plugin to $MY_PKG_NAME"
    mv -v template_basic_plugin/ $MY_PKG_NAME
else
    echo "Package already renamed"
fi

# This will remove references to our upstream repository for safety's sake
cd $MY_PKG_DIR
git remote remove origin 2> /dev/null || true

In [ ]:
%%bash

echo $MY_PKG_NAME

### Update the Package Name

Now, we can use `ls` to look around in your package directory.


In [ ]:
%%bash

cd $MY_PKG_DIR
ls --color

This package is set up to be an installable Python package named "my_package". Let's update it to install as "cool_plugins" instead.

Rename the default plugin package directory to "cool_plugins", then call `tree` to see the entire directory structure.

In [ ]:
%%bash

cd $MY_PKG_DIR

# Rename the package directory
if [[ -d "my_package" ]]; then
    echo "Moving my_package to $MY_PKG_NAME"
    git mv my_package $MY_PKG_NAME
else
    echo "Already moved"
fi

# Show the directory tree two-levels deep
tree -L 2

In [ ]:
%%bash

cd $MY_PKG_DIR/cool_plugins
tree

![plugins directory structure](./images/plugins_directory_structure.png)

### Update Pertinent Files

1. Update README.md (`vim README.md`)
  - Find/replace all occurrences of @package@  with your package name 
  - Vim Tip :%s/@package@/cool_plugins/g 
  
**Note**: The @ symbols are for ease of searching, take them out when you put your 
package name in!

2. Update pyproject.toml (`vim pyproject.toml`, more on this in soon.)
  - Find/replace all occurrences of my_package  with your package name 

3. Add and commit your changes.

#### Note

To save time at this point in the tutorial, we will copy these updated files
into place.

In [ ]:
"""Overwrite cool_plugins' pyproject.toml and README.md with correct contents."""

import os

with open("./updated_files/pyproject.toml", "r") as rf:
    toml_lines = rf.readlines()

with open(f"{os.environ['GEOIPS_PACKAGES_DIR']}/cool_plugins/pyproject.toml", "w") as wf:
    wf.writelines(toml_lines)

with open("./updated_files/README.md", "r") as rf:
    md_lines = rf.readlines()

with open(f"{os.environ['GEOIPS_PACKAGES_DIR']}/cool_plugins/README.md", "w") as wf:
    wf.writelines(md_lines)

In [ ]:
%%bash

cd $MY_PKG_DIR
git add README.md pyproject.toml
git commit -m "Updated name of template plugin package to mine" || true

#### Install your package 
Now that your package is updated, you can install it! 

We'll use `pip install -e $MY_PKG_DIR` where `-e` means "editable". This installs the package in "editable" mode so we can edit the package after it is installed and changes will be reflected in the installed package.

*For those who are interested, this acts similarly to a symlink in Linux, but has some more complexity behind it.*

In [ ]:
%%bash

pip install -e $MY_PKG_DIR

### See what you just installed 

Use the GeoIPS CLI to list the installed packages. Yours should be there!

In [ ]:
%%bash

geoips list packages

### A bit about pyproject.toml

Installing Python packages requires metadata that describes the package and how to 
install it. 

`pyproject.toml` defines this information for pip, including: 
  - Package name, version, description, license, etc. 
  - Which files should be contained in the package when installed 
  - How to build the package 

We make GeoIPS aware of our package using the `geoips.plugin_packages` namespace.
This allows GeoIPS to find all plugins within packages registered to this namespace.

GeoIPS automatically identifies all plugins defined within a plugin package via a plugin
registry. You can manually create these files via `geoips config create-registries`, 
however, GeoIPS will automatically create these files if a requested plugin cannot be
found. This usually occurs the first time GeoIPS is initialized.

**NOTE** for plugin registries to write successfully: 

1. All installed  plugin names within a given interface must be unique 

2. All installed plugins must be formatted and defined correctly 

We will make use of this more later! Modify plugin template solutions on 
[NRLMMD github.com](https://github.com/NRLMMD-GEOIPS/template_basic_plugin/tree/workshop-2023-solutions).

```
[tool.poetry.plugins."geoips.plugin_packages"]
"cool_plugins" = "cool_plugins" 
``` 

## GeoIPS Command Line Interface (CLI) Tutorial

The GeoIPS CLI can provide you with a lot of information about the installed plugin packages and their plugins. Please follow this [Jupyter Notebook](./CLI_Tutorial.ipynb) for instructions on how to 
make use of the GeoIPS CLI.

## Hands on: Create a Workflow Plugin (workflows  YAML-based interface) 

Solutions: **DO NOT VIEW UNTIL END OR IF STUCK**

Visit here for [solutions](https://github.com/NRLMMD-GEOIPS/geoips_tutorials/tree/2026-workshop-updates/solutions)

## New Workflow Plugins

In the following section we'll walk through the contents of a workflow plugin. At their
core, they are essentially a configuration file containing a series of sequential steps
instructing the OBP what plugins to call and in what order. A step can depend on 
previous steps in the case that they need data and/or metadata from one or more plugins.

Under the hood, the OBP constructs an xarray.DataTree object which is a hierarchical 
data container that preserves its entries upon order of insertion. It contains data and
metadata produced from each step, unless the data from a given step is no longer needed
downstream. Think of it as a data / metadata flowchart.

In [ ]:
%%bash

geoips describe workflow Infrared

In [ ]:
%%bash

geoips expand Infrared --color

## Run the Infrared Workflow

Now that we've seen the order of steps in an example workflow, let's run it and see
what it produces.

**NOTE:** you can execute a workflow in two different fashions. They are 

``geoips run order_based <args>`` 

OR

``geoips test workflow <workflow_type>``

``geoips run order_based`` should be used when using custom arguments for a specific 
workflow. For reproducibility and testing purposes, you can run 
``geoips test workflow`` which will override a given workflow with the contents 
specified in that workflow's ``test`` section. This will use the hardcoded set of 
arguments specified in ``test`` so you don't have to specify them at the commandline 
every time you want to execute that workflow.

For our use case, let's use ``geoips test workflow``.

In [ ]:
%%bash

geoips test workflow Infrared

In [ ]:
import os

from IPython.display import Image

# Provide the path to your PNG file
Image(f"{os.environ['GEOIPS_OUTDIRS']}/infrared_full_run.png")

## Hands On: Modify the Example Workflow

Now that we understand how a workflow operates, let's modify it. Change the sector in
the previous workflow to something different. Keep in mind that this is data from GOES16,
so you'll need a sector that is located within the bounds of the North / South American
region.

In [ ]:
%%bash

geoips list sectors
# Here is a listing of available sectors you can choose

## Restart Your Kernel

Since we installed a new python package we need to restart the JupyterLab Kernel so that
the python environment recognizes it.

In [ ]:
from importlib.resources import files
from pathlib import Path

from workshop_utils import yaml_editor

# NOTE:
#   CHANGE THE NAME OF THE WORKFLOW PLUGIN TO 'workshop_infrared'
#   CHANGE THE NAME OF THE 'sector' STEP TO AN APPLICABLE SECTOR

yaml_editor(
    f"{files('cool_plugins') / 'plugins/yaml/workflows'}/workshop_infrared.yaml",
    default_yaml=Path(
        f"{files('geoips') / 'plugins/yaml/workflows/unit_tests/Infrared.yaml'}"
    ).read_text(encoding="utf-8"),
)

In [ ]:
from importlib.resources import files
from pathlib import Path

import yaml

# PREVIEW YOUR UPDATED WORKFLOW

config_path = Path(
    f"{files('cool_plugins') / 'plugins' / 'yaml' / 'workflows'}/workshop_infrared.yaml"
).expanduser()
with config_path.open() as stream:
    config = yaml.safe_load(stream)

In [ ]:
%%bash

# uncomment the line below if this code cell fails

# geoips config create-reg

geoips test workflow workshop_infrared

In [ ]:
import os

from IPython.display import Image

# Provide the path to your PNG file
Image(f"{os.environ['GEOIPS_OUTDIRS']}/infrared_full_run.png")

### GeoIPS Yaml-based plugin properties

All YAML plugins will begin with these same four properties as shown below.

In the image on the right, we demonstrate the top-level fields of a workflow plugin, the
newest plugin type in GeoIPS that is used by the Order Based Procflow (OBP).

![Workflow top level keys](./images/workflow_top_level.png)

### First Workflow Plugin: Severe Storms RGB imagery from G16 ABI data

The following sections demonstrate how to create a new workflow plugin.

In [ ]:
from importlib.resources import files

from workshop_utils import yaml_editor

# NOTE:
#   NAME THIS WORKFLOW 'My-ABI-Severe-Storms

yaml_editor(
    f"{files('cool_plugins') / 'plugins/yaml/workflows'}/My-ABI-Severe-Storms.yaml",
    default_yaml="""\
interface: workflows
family: order_based
name:
docstring: |
  ABI Severe Storms RGB workflow.
spec:
  steps:
    """
)

## Adding New Steps

The following section should implement the required steps to produce a severe storms
algorithm. You'll specify it yourself. We'll help if needed!

Any Step ID should be a valid python identifier. Additionally, it should attempt to 
describe what the step does. I.e.

```yaml
read_abi_data:
  kind: reader
  name: abi_netcdf
  depends_on: [retrieve_area_definition]
  arguments:
    chans: &variable-list ['B08BT', 'B10BT', 'B07BT', 'B13BT', 'B05Ref', 'B02Ref']
```

Steps:

Every step MUST specify ``kind`` and ``name``. ``kind`` is the type of plugin being 
applied, but singular. I.e. ``algorithms`` kind == ``algorithm``. ``name`` should be
the name of the plugin that is type ``kind``.

1. Sector:
    This step should use the `conus` sector. It has no dependencies. Make sure to
    name ID of this step ``retrieve_area_definition``. The reader will depend on this
    step, otherwise this workflow is VERY slow, as geolocation will need to be 
    calculated for LOW, MID, and HIGH resolutions.
2. Reader:
    This step should read the channels needed to produce the Severe Storms RGB from 
    GOES ABI data. For demonstration purposes, we've defined that above.
3. Interpolator:
    An interpolator step always has two dependencies. A sector step and a data step.
    Dependencies should reference their corresponding Step ID. Considering we've only
    defined two steps so far, add those as this steps dependency. We will use the 
    ``interp_nearest`` interpolator plugin. Additionally, an interpolator must specify
    the ``varlist`` arguments. In this case, it should map 1:1 to the variables read
    by the reader. You can reduce complication by specifying ``varlist: *variable-list``.
4. Algorithm:
    Apply the ``severe_storms`` algorithm plugin. There is no need to specify dependencies
    as GeoIPS automatically makes the previous step a dependency to the next step if
    not specified. It takes the following arguments:
    ```yaml
    red:
      range: [-35.0, 5.0]
      units: Kelvin
      gamma: 1
    green:
      range: [5.0, 60.0]
      units: Kelvin
      gamma: 0.5
    blue:
      range: [-75.0, 25.0]
      units: percentage
      gamma: 1
    ```
5. Colormap:
    Utilizes the ``cmap_rgb`` colormapper plugin. No dependencies.
8. Output Formatter:
    Output formatter plugins depend on multiple previous steps. In our case, the 
    output formatter step depends on the output of our algorithm, colormapper, and 
    sector plugin. It utilizes the ``imagery_clean`` output formatter plugin. It takes 
    the following arguments:
    ```yaml
    arguments:
      product_name: My-ABI-Severe-Storms
      output_fnames: [!ENV $GEOIPS_OUTDIRS/abi_severe_storms.png]
    ```

Using the instructions specified above, write all of the steps for the 
``My-ABI-Severe-Storms`` workflow.

In [ ]:
from importlib.resources import files

from workshop_utils import yaml_editor

# NOTE:
#   NAME THIS WORKFLOW 'My-ABI-Severe-Storms

yaml_editor(
    f"{files('cool_plugins') / 'plugins/yaml/workflows'}/My-ABI-Severe-Storms.yaml",
    default_yaml="",
)

### Update the Plugin Registry

Let's use the CLI to get more information about the plugin we just created.

### Describing plugins

To get more information about a particular plugin (or interface), call `geoips describe`. This is the generic format to follow via the CLI:

`geoips describe <interface_name> <source_name>.<plugin_name>`

In [ ]:
%%bash

# Uncomment the line below if this code cell fails

# geoips config create-reg

geoips describe workflow My-ABI-Severe-Storms

## Adding a New Algorithm Plugin

You might not have noticed this before, but one of the plugin steps we referenced in our new workflow doesn't actually exist. This is the step that is type ``kind: algorithm`` step which references a ``severe_storms`` algorithm. If you run ``geoips list algorithms`` you'll see that no such algorithm exists. We are going to create that plugin right now.

In [ ]:
%%bash

cd $MY_PKG_DIR/$MY_PKG_NAME/plugins
mkdir -p classes/algorithms

### The same three top-level attributes
Just like the Yaml-based plugins, this class-based plugin has three top-level attributes that are common to all GeoIPS plugins: interface, family, and name. Here:
- `interface` is "algorithms" to indicate that this plugin belongs to the Algorithms interface.
- `family` is "xarray_to_numpy", a common family for algorithms indicating that it accepts an Xarray DataSet as input and returns numpy ndarray as output.
- `name` is the name of the algorithm (which we will update).

![Updating top level portions of new algorithm](./images/alg_screenshot.png)

### Let's update our `severe_storms` algorithm

We'll provide the following as a base template. Just click `validate and save`.

In [ ]:
from importlib.resources import files

from workshop_utils import python_editor

python_editor(
    f"{files('cool_plugins') / 'plugins/classes/algorithms/severe_storms.py'}",
    default_python='''\
"""Severe Storms WMO RGB recipe."""

from geoips.interfaces.class_based.algorithms import BaseAlgorithmPlugin

import logging

LOG = logging.getLogger(__name__)


class SevereStormsAlgorithmPlugin(BaseAlgorithmPlugin):
    """Severe Storms WMO RGB recipe."""

    interface = "algorithms"
    family = "xarray_to_numpy"
    name = "severe_storms"
''',
)

### Updating your Algorithm

All module and class-based plugins, including Algorithms, must include a `call()` function. This function is what is called when the plugin is executed. The `call()` function's signature is determined by the algorithm's family.

We will largely trim what arguments are present in this plugin's call signature. All we need are the ``red``, ``green``, and ``blue`` argument dictionaries we defined in our workflow.

Copy the following into your new plugin class.

```python
    def call(self, xobj, red, green, blue):
        """Apply WMO's Severe Storms RGB recipe.

        Parameters
        ----------
        xobj : xarray.Dataset
            The dataset containing variables needed for the severe storms rgb recipe.
        red : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the red gun of a RGB recipe.
        green : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the green gun of a RGB recipe.
        blue : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the blue gun of a RGB recipe.

        Returns
        -------
        numpy.ndarray
            numpy.ndarray or numpy.MaskedArray of qualitative RGBA image output
        """
        rparams = red
        gparams = green
        bparams = blue
```

In [ ]:
from importlib.resources import files

from workshop_utils import python_editor

python_editor(
    f"{files('cool_plugins') / 'plugins/classes/algorithms/severe_storms.py'}",
    default_python="",
)

## Apply Severe Storms RGB Recipe

The following image displays the recipe for the Severe Storms RGB using GOES ABI data.

![GOES ABI Severe Storms RGB Recipe](./images/goes_abi_severe_storms.png)

For reference, here is the mapping for GeoIPS ABI variables to ABI sensor channels:

```python
GOES_ABI = {
    "LOW": [
        "B04Rad",
        "B04Ref",  # 1.37um Near-IR Cirrus
        "B06Rad",
        "B06Ref",  # 2.2um  Near-IR Cloud Particle Size
        "B07Rad",
        "B07BT",  # 3.9um  IR      Shortwave Window
        "B08Rad",
        "B08BT",  # 6.2um  IR      Upper-level tropospheric water vapor
        "B09Rad",
        "B09BT",  # 6.9um  IR      Mid-level water vapor
        "B10Rad",
        "B10BT",  # 7.3um  IR      Lower-level Water Vapor
        "B11Rad",
        "B11BT",  # 8.4um  IR      Cloud-top phase
        "B12Rad",
        "B12BT",  # 9.6um  IR      Ozone
        "B13Rad",
        "B13BT",  # 10.3um IR      Clean IR Longwave Window
        "B14Rad",
        "B14BT",  # 11.2um IR      IR Longwave window
        "B15Rad",
        "B15BT",  # 12.3um IR      Dirty Longwave Window
        "B16Rad",
        "B16BT",  # 13.3um IR      CO2 Longwave infrared
    ],  
    "MED": [
        "B01Rad",
        "B01Ref",  # 0.47um Vis     Blue
        "B02Rad",
        "B02Ref",  # 0.64um Vis     Red
        "B05Rad",
        "B05Ref",  # 1.6um  Near-IR Snow/Ice
    ],  
    "HIGH": ["B03Rad", "B03Ref"],  # 0.86um Near-IR Veggie
}
```

In this case, since we're using:

```yaml
red:
  equation: WV6.2 - WV7.3 (Kelvin)
green:
  equation: IR3.9 – IR10.3 (Kelvin)
blue:
  equation: NIR1.6 - VIS0.64 (%)
```

We'll want to convert these to GeoIPS nomenclature. Kelvin is a brightness temperature
and percentage (%) is in reflectance units.

Try your hand at finding the correct variables and writing the correct equations in your
algorithm.


### Update the call function's functionality

Inside the `call()` is where the actual data manipulation occurs. To update the algorithm to produce Severe Storms RGB from input data, replace the contents of the `call()` function with the following. If you examine the code below, you will see that it:
- Extracts the relevant variables from the xarray object.
- Applies the appropriate equation for each RGB gun
- Converts input data to expected units
- Crops data to its appropriate min / max range
- Applies gamma corrections to each RGB gun
- Returns the data as an ``[[r], [g], [b], [a]]`` np.ndarray

The returned will now be a Severe Storms RGBA tuple.

Add the following contents to your plugin's ``call()`` function.

```python

        # NOTE: Uncomment the lines below and replace the <clr_geoips_varX> portions
        # with a string of the correct GeoIPS variable from the GEOIPS_ABI variable
        # dictionary listed above

        # red = xobj[<red_geoips_var1>].to_masked_array() - xobj[<red_geoips_var2>].to_masked_array()
        # grn = xobj[<grn_geoips_var1>].to_masked_array() - xobj[<grn_geoips_var2>].to_masked_array()
        # blu = xobj[<blue_geoips_var1>].to_masked_array() - xobj[<blue_geoips_var2>].to_masked_array()

        # Ensure red and green guns are in Kelvin units
        from geoips.data_manipulations.conversions import unit_conversion

        red = unit_conversion(red, input_units="Kelvin", output_units=rparams["units"])
        grn = unit_conversion(grn, input_units="Kelvin", output_units=gparams["units"])
        # No unit conversion needed to be applied to blue gun

        from geoips.data_manipulations.corrections import apply_data_range, apply_gamma

        data_range = rparams["range"]
        gamma = rparams["gamma"]
        red = apply_data_range(
            red,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        red = apply_gamma(red, gamma)

        data_range = gparams["range"]
        gamma = gparams["gamma"]
        grn = apply_data_range(
            grn,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        grn = apply_gamma(grn, gamma)

        data_range = bparams["range"]
        gamma = bparams["gamma"]
        blu = apply_data_range(
            blu,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        blu = apply_gamma(blu, gamma)

        from geoips.image_utils.mpl_utils import (
            alpha_from_masked_arrays,
            rgba_from_arrays,
        )

        alp = alpha_from_masked_arrays([red, grn, blu])
        rgba = rgba_from_arrays(red, grn, blu, alp)

        return rgba


# Tells pluginify (plugin registry package) what object is the actual plugin we want to
# add to the registry
PLUGIN_CLASS = SevereStormsAlgorithmPlugin
```

In [ ]:
from importlib.resources import files

from workshop_utils import python_editor

python_editor(
    f"{files('cool_plugins') / 'plugins/classes/algorithms/severe_storms.py'}",
    default_python="",
)

In [ ]:
%%bash

# Uncomment the line below if this code cell fails

# geoips config create-reg

geoips describe algorithm severe_storms

### Use your new workflow!

Now that we've created the missing algorithm plugin, the workflow we created is able
to be ran. We'll do this via the ``geoips run`` command.

- GeoIPS is called via a command line interface (CLI), as we've shown previously.
- The main command that you will use is ``geoips run``, which will run your 
  data through the order based procflow (OBP) using the specified plugins.
- It's easiest to do this via a script, and scripts are stored in your plugin package's 
  `tests/` directory because they can be used later to regression test your package 
- Since we're running this in a notebook, we can easily run these commands with a code cell!

In [1]:
%%bash

geoips run order_based My-ABI-Severe-Storms $GEOIPS_TESTDATA_DIR/test_data_abi/data/goes16_20200918_1950/*

22_183019      readers.py:158  WARNING: 'chans' is deprecated and will be removed in GeoIPS 2.0. Use 'variables' instead.
22_183019    log_setup.py:267     INFO: USER: evan
22_183019    log_setup.py:268     INFO: HOST: spring
22_183019      readers.py:158  WARNING: 'chans' is deprecated and will be removed in GeoIPS 2.0. Use 'variables' instead.
22_183019    log_setup.py:161  INTERACTIVE: Begin processing 'My-ABI-Severe-Storms' workflow.
22_183020    log_setup.py:161  INTERACTIVE: Running step 'retrieve_area_definition': sector plugin 'conus'
22_183020     workflow.py:601     INFO: Calling step 'retrieve_area_definition' with arguments: {}
22_183020    log_setup.py:161  INTERACTIVE: Completed step 'retrieve_area_definition': sector plugin 'conus'
22_183020    log_setup.py:161  INTERACTIVE: Running step 'read_abi_data': reader plugin 'abi_netcdf'
22_183020     workflow.py:601     INFO: Calling step 'read_abi_data' with arguments: {'chans': 'list[6]'}
22_183020      readers.py:83   WARNI

### Viewing the log output

This will write some log output.  If your script succeeded it will have a line near the
end with:
`IMAGESUCCESS wrote <image_path>.png`.

To view your output, look for than line and open the hyperlinked image (or run the cell below).

If successful, the output image should look like this:

![ABI Severe Storms RGB](./images/abi_severe_storms.png)

In [ ]:
import os

from IPython.display import Image

Image(f"{os.environ['GEOIPS_OUTDIRS']}/abi_severe_storms.png")

### Create your own algorithm for a new RGB recipe

Now that we've walked through how to create a workflow and algorithm plugin, try your
hand at creating a new workflow plugin (or modifying the original) using a newly created
algorithm, by you. 

Go ahead and choose an RGB recipe from the following WMO RGB report to create a new 
algorithm for.

**(RGBs begin at page 31) [RGB-WS-2025_MeetingReport_final.pdf](../docs/source/WMO_RGBs/RGB-WS-2025_MeetingReport_final.pdf)**

Additionally, you'll need to how the channels for the ABI sensor map to the `abi_netcdf`
reader plugin. Reference the following file for that. If you reference the previous
algorithm you created, you can map GeoIPS channels to ABI sensor channels shown in the
PDF to reflect this process. (I.e `WV 6.2 - WV 7.3 == B08BT - B10BT`). 

Make sure to note the units of each RGB gun as well.

**[geoips_reader_channel_mapping.py](../solutions/geoips_reader_channel_mapping.py)**

## Continue with the next tutorial

Nice job completing this! I'd recommend you move on to the [output formatters tutorial](./output_formatters_tutorial.ipynb) now.